# Extended Kalman Filter, A Revisit on the Non-Linear State Space Model

In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
import numpyro
numpyro.set_host_device_count(4)
import jax
import jax.numpy as jnp
import arviz as az


from numpyro import distributions as dist
from numpyro.infer import MCMC, NUTS

import matplotlib.pyplot as plt
from jax import random

from bunobee.models.ssp.kalman_1d_ekf import (
    kalman_filter_1d_ekf,
    kalman_rts_smoother_1d_ekf,
 )
from bunobee.models.ssp import plot_states, transform_to_ekf
from bunobee.models.ssp.utils import construct_states_prior, posterior_to_xarray

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
# False → plain Kalman filter; True → enforce a_obs at sampled windows
USE_DISCLOSURE = True 
# number of disclosure windows to sample
N_PERIODS      = 3    
# consecutive steps per window
N_POINTS       = 10     
# disclose noise variance
DISC_OBS_SIGMA = 0.01

# for EKF positivity constraint
EXPONENT = 0.5


In [ ]:
df = pd.read_csv(
    '../resource/simple/df.csv', parse_dates=['date']
)
# for quick testing
df = df[-365:].reset_index(drop=True)

saturation_df = pd.read_csv(
    '../resource/simple/saturation_df.csv',
)
coefs_df = pd.read_csv(
    '../resource/simple/coefs_df.csv',
)

regressors = saturation_df["regressor"].tolist()
response = df["sales"].values

response_norm = response.mean()
y = np.log(response / response_norm)
y = jnp.array(y)
X = df[regressors].values
sat_arr = saturation_df.set_index("regressor").loc[regressors, "saturation"].values
X = np.log1p(X / sat_arr)
sdx = X.std(0)
sdy = y.std()
sdy_over_sdx = sdy / sdx
Z = jnp.concatenate([jnp.ones((X.shape[0], 1)), jnp.array(X)], -1)

positivity_idx = jnp.array([0] + [1] * len(regressors))
positivity_idx = positivity_idx.astype(bool)

print("y shape", y.shape)
print("Z shape", Z.shape)
print("X shape", X.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df["date"], y, label="y")

In [ ]:
n_steps = y.shape[0]
n_states = Z.shape[-1]
state_labels = ["intercept"] + regressors
ssp_priors = None

if coefs_df is not None:
    coefs_lookup = coefs_df.set_index("regressor")["coef"]
    # natural-scale true state: intercept linear, channel coefficients in λ-space
    true_state_nat = jnp.array([0.0] + [coefs_lookup[r] for r in regressors])

    if USE_DISCLOSURE:
        ssp_priors = construct_states_prior(
            n_steps=n_steps,
            n_states=n_states,
            true_states=true_state_nat,
            regressors=regressors,
            n_periods=N_PERIODS,
            n_points=N_POINTS,
            obs_scale=DISC_OBS_SIGMA,
        )
        print("a_obs shape:", ssp_priors["a_obs"].shape)
        print("P_obs shape:", ssp_priors["P_obs"].shape)
        print("n disclosed steps:", ssp_priors.sizes["obs_point"])
    else:
        print("disclosure disabled")
else:
    print("no ground truth state available; disclosure disabled")


In [ ]:
state_labels = ["intercept"] + regressors
n_regressors = len(regressors)
n_states = 1 + n_regressors

# Per-state Beta(α, β) on a base in [0, 1], scaled to natural-scale σ_q:
#   sigma_q_nat = Beta(α, β) * scale_nat
# Mirrors v2's `Beta(2, 10) * sdy(_over_sdx) * 0.1` parameterisation but
# keyed per-state so each regressor carries its own sdy_over_sdx scale.
alpha_per_state = np.full(n_states, 2.0)
beta_per_state  = np.full(n_states, 10.0)
scale_per_state = np.concatenate([
    np.array([sdy * 0.1]),
    np.asarray(sdy_over_sdx) * 0.1,
])

base_ds = xr.Dataset(
    {
        "a0":                  (("state",),     np.array([0.0] + [0.1] * n_regressors)),
        "P0":                  (("state",),     np.array([1.0] + [0.1] * n_regressors)),
        "sdy":                 float(sdy),
        "sdy_over_sdx":        (("regressor",), np.asarray(sdy_over_sdx)),
        "sigma_q_alpha_prior": (("state",),     alpha_per_state),
        "sigma_q_beta_prior":  (("state",),     beta_per_state),
        "sigma_q_scale_prior": (("state",),     scale_per_state),
    },
    coords={
        "state":     state_labels,
        "regressor": regressors,
    },
    attrs={"sigma_q_family": "beta"},
)

if ssp_priors is not None:
    ssp_priors = xr.merge([base_ds, ssp_priors])
else:
    null_disc = xr.Dataset(
        {
            "a_obs":          (("time", "state"), np.zeros((n_steps, n_states))),
            "P_obs":          (("time", "state"), np.full((n_steps, n_states), np.inf)),
            "obs_idx":        (("obs_point",),    np.array([], dtype=int)),
            "positivity_idx": (("state",),        np.array([False] + [True] * n_regressors)),
        },
        coords={
            "time":      np.arange(n_steps),
            "state":     state_labels,
            "obs_point": np.array([], dtype=int),
        },
    )
    ssp_priors = xr.merge([base_ds, null_disc])

ssp_priors = ssp_priors.assign_attrs({"filter": "kalman_1d", "sigma_q_family": "beta"})

print("----- Natural-scale priors -----")
print(ssp_priors)


In [ ]:
ssp_priors_ekf = transform_to_ekf(ssp_priors, exponent=EXPONENT, match="median")
print("\n----- EKF-scale priors -----")
print(ssp_priors_ekf)

In [ ]:
# def test_run(ssp_priors, exponent=EXPONENT):
#     a0      = jnp.array(ssp_priors_ekf["a0"].values)
#     # EKF expects diagonal variances; collapse the full a-space covariance.
#     P0    = jnp.array(ssp_priors_ekf["P0"].values)
#     a_obs   = jnp.array(ssp_priors_ekf["a_obs"].values)
#     P_obs   = jnp.array(ssp_priors_ekf["P_obs"].values)
#     obs_idx = np.asarray(ssp_priors_ekf["obs_idx"].values)

#     print("a0 shape:", a0.shape)
#     print(f"  a0[0] = {a0[0]:.3f}  (intercept, linear space)")
#     print(
#         f"  a0[1] = {a0[1]:.3f}  (channel, a-space → λ₀ = exp({EXPONENT}·a0) = "
#         f"{jnp.exp(EXPONENT * a0[1]):.3f})"
#     )
#     print("P0:", P0)

#     sigma_h = jnp.array(0.1)
#     # using prior loc as a test value
#     sigma_q_raw = jnp.array(ssp_priors["sigma_q_loc_prior"].values)
#     sigma_q = jnp.concatenate([sigma_q_raw[:1], jnp.repeat(sigma_q_raw[1:], Z.shape[-1] - 1)])
#     print("sigma_q expanded:", sigma_q.shape, sigma_q)

#     lp, at, Pt, _, _, _ = kalman_filter_1d_ekf(
#         a0=a0,
#         P0=P0,
#         sigma_h=sigma_h,
#         sigma_q=sigma_q,
#         y=y,
#         Z=Z,
#         logp=True,
#         exponent=EXPONENT,
#         positivity_idx=positivity_idx,
#     )

#     # recover intensities: intercept stays as-is, channels via exp(EXPONENT·a)
#     lam = jnp.where(positivity_idx, jnp.exp(jnp.clip(exponent * at, -10.0, 10.0)), at)

#     print("lp:", lp)
#     print("at shape:", at.shape)
#     print("intercept at[-1,0]:", at[-1, 0], "  ← linear space")
#     print(f"channel lam[-1,1]: {lam[-1, 1]}  ← λ = exp({exponent}·a)")

# test_run(ssp_priors_ekf, positivity_idx)


In [ ]:
ssp_priors_ekf["sigma_q_scale_prior"]

In [ ]:
def make_nuts_fn(ssp_priors_ekf, y, Z, exponent):
    a0    = jnp.array(ssp_priors_ekf["a0"].values)
    P0    = jnp.array(ssp_priors_ekf["P0"].values)
    a_obs = jnp.array(ssp_priors_ekf["a_obs"].values)
    P_obs = jnp.array(ssp_priors_ekf["P_obs"].values)
    # a-space Beta hyperparameters: α, β unchanged from natural scale; only
    # `scale` was rescaled inside transform_to_ekf via lognormal mode-matching.
    alpha_a = jnp.array(ssp_priors_ekf["sigma_q_alpha_prior"].values)
    beta_a  = jnp.array(ssp_priors_ekf["sigma_q_beta_prior"].values)
    scale_a = jnp.array(ssp_priors_ekf["sigma_q_scale_prior"].values)
    positivity_idx = jnp.array(ssp_priors_ekf["positivity_idx"].values).astype(bool)

    def nuts_fn():
        # support [0, 1], mode 0.1
        sigma_h_base = numpyro.sample(
            "sigma_h_base",
            dist.Beta(2.0, 10.0),  
        )
        sigma_h = numpyro.deterministic(
            "sigma_h",
            sigma_h_base * sdy,
        )

        # Split level (intercept) and regressors so each block keeps its own
        # Beta hyperparameters; recombine into the per-state sigma_q vector.
        sigma_q_lev = numpyro.sample(
            "sigma_q_lev_base", dist.Beta(alpha_a[0], beta_a[0]) 
        )
        sigma_q_reg = numpyro.sample(
            "sigma_q_reg_base", dist.Beta(alpha_a[1:], beta_a[1:])
        )
        sigma_q = numpyro.deterministic(
            "sigma_q",
            jnp.concatenate([sigma_q_lev[None], sigma_q_reg]) * scale_a,
        )

        lp, at, Pt, vt, Ft, Kt = kalman_filter_1d_ekf(
            a0=a0,
            P0=P0,
            sigma_h=sigma_h,
            sigma_q=sigma_q,
            y=y,
            Z=Z,
            logp=True,
            exponent=exponent,
            positivity_idx=positivity_idx,
            a_obs=a_obs,
            P_obs=P_obs,
        )
        lam = jnp.where(positivity_idx, jnp.exp(jnp.clip(exponent * at, -10.0, 10.0)), at)

        numpyro.factor("lp", lp)
        numpyro.deterministic("at", at)
        numpyro.deterministic("Pt", Pt)
        numpyro.deterministic("lam", lam)
        numpyro.deterministic("mu", jnp.sum(Z * lam, -1))

    return nuts_fn

In [ ]:
nuts_fn = make_nuts_fn(ssp_priors_ekf, y, Z, exponent=EXPONENT)
nuts_kernel = NUTS(nuts_fn)
mcmc = MCMC(
    nuts_kernel,
    num_warmup=400,
    num_samples=400,
    num_chains=4,
)
rng_key = random.PRNGKey(0)
rng_keys = random.split(rng_key, 2)
rng_key = rng_keys[0]
mcmc_rng_key = rng_keys[1]
mcmc.run(mcmc_rng_key)

In [ ]:
# Group by chain so every site has leading (chain, draw) axes — matches v2's pattern
# and is required by posterior_to_xarray.
numpyro_posteriors = mcmc.get_samples(group_by_chain=True)

# ── Post-MCMC RTS smoother ────────────────────────────────────────────────
# RTS only needs (at, Pt, sigma_q); all three are already deterministics on
# the MCMC posterior, so no filter rerun is required.
at_draws      = numpyro_posteriors["at"]       # (n_chains, n_draws, T, n_states)
Pt_draws      = numpyro_posteriors["Pt"]
sigma_q_draws = numpyro_posteriors["sigma_q"]  # (n_chains, n_draws, n_states)

at_smooth, Pt_smooth = jax.vmap(jax.vmap(
    lambda at_i, Pt_i, sigma_q_i: kalman_rts_smoother_1d_ekf(
        at=at_i,
        Pt=Pt_i,
        sigma_q=sigma_q_i,
    )
))(at_draws, Pt_draws, sigma_q_draws)

lam_smooth = jnp.where(
    positivity_idx[None, None, None, :],
    jnp.exp(jnp.clip(EXPONENT * at_smooth, -10.0, 10.0)),
    at_smooth,
)
mu = jnp.einsum("cdts,ts->cdt", lam_smooth, Z)

print(f"RTS at_smooth shape: {at_smooth.shape}")

# Smoother results flow straight into xarray; caller owns the dim/coord schema.
posterior = posterior_to_xarray(
    {
        **numpyro_posteriors,
        "at_smooth":  np.asarray(at_smooth),
        "Pt_smooth":  np.asarray(Pt_smooth),
        "lam_smooth": np.asarray(lam_smooth),
        "mu":         np.asarray(mu),
    },
    dims={
        "sigma_q_base": ["state"],
        "sigma_q":      ["state"],
        "at":           ["time", "state"],
        "Pt":           ["time", "state"],
        "lam":          ["time", "state"],
        "at_smooth":    ["time", "state"],
        "Pt_smooth":    ["time", "state"],
        "lam_smooth":   ["time", "state"],
        "mu":           ["time"],
    },
    coords={
        "state": state_labels,
        "time":  df["date"].values,
    },
    drop=["sigma_h_base", "sigma_q_lev", "sigma_q_reg"],
)

idata = az.InferenceData(posterior=posterior)
az.summary(idata, var_names=["sigma_h", "sigma_q"])

In [ ]:
a_obs_plot   = np.asarray(ssp_priors["a_obs"].values) if USE_DISCLOSURE else None
P_obs_plot   = np.asarray(ssp_priors["P_obs"].values) if USE_DISCLOSURE else None
obs_idx_plot = np.asarray(ssp_priors["obs_idx"].values)   if USE_DISCLOSURE else None

plot_states(
    posterior,
    df["date"].values,
    state_labels,
    states_key=["lam", "lam_smooth"],
    colors=["C0", "C1"],
    coefs_df=coefs_df,
    obs_idx=obs_idx_plot,
    a_obs=a_obs_plot,
    P_obs=P_obs_plot,
    title=(
        f"Log-normal EKF — λ_t = exp({EXPONENT}·a_t)  [disclosure ON]"
        if USE_DISCLOSURE
        else f"Log-normal EKF — λ_t = exp({EXPONENT}·a_t)  [disclosure OFF]"
    ),
)
plt.show()

In [ ]:
# Flatten (chain, draw) → samples for predictive sampling.
mu_samples      = posterior["mu"].stack(sample=("chain", "draw")).transpose("sample", "time").values
sigma_h_samples = posterior["sigma_h"].stack(sample=("chain", "draw")).values

eps_samples = np.random.normal(
    0,
    sigma_h_samples[:, None],
    size=mu_samples.shape,
)

# compute p5, p50, p95 for forecast
yhat_lower, yhat_mid, yhat_upper = np.quantile(np.exp(mu_samples + eps_samples) * response_norm, q=[0.05, 0.5, 0.95], axis=0)

n = 100
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(np.arange(n), response[-n:], label='Observed', alpha=0.5, color="orange", s=10)
ax.plot(np.arange(n), yhat_mid[-n:], label='Forecast', linestyle='--', alpha=0.8, color="dodgerblue")
ax.fill_between(np.arange(n), yhat_lower[-n:], yhat_upper[-n:], alpha=0.5, label='95% Prediction Interval', color="dodgerblue")
ax.set_title('State Space Model Forecast')
